In [1]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Aggregate targets to output resolution

In [2]:
def build_targets():
    df = pd.read_parquet(variables.TARGET_DATASET_PATH)
    s =variables.PIPELINE_START_DATE
    e =variables.PIPELINE_END_DATE + pd.Timedelta(days=2) # Add an extra 2 days to cover the last horizons of the range (assuming max 48 hours)
    df = df[(df.index>=s) & (df.index <= e)]
    # Selected just the target column
    idx = df.index
    df = df[variables.SELECTED_TARGET_COLUMN_NAME].values.astype(np.float32)

    # Derive target horizons
    df_shifted = np.empty((len(df), variables.HORIZON_COUNT), dtype=np.float32)
    for h in tqdm(range(variables.HORIZON_COUNT), desc="shifting horizons", leave=False):
        df_shifted[:len(df) - h, h] = df[h:]
        df_shifted[(len(df)) - h:, h] = np.nan

    df_shifted = pd.DataFrame(df_shifted,index=idx,columns=[f"target_h{h + 1}" for h in range(variables.HORIZON_COUNT)],)
    
    del df
    

    return df_shifted.loc[:variables.PIPELINE_END_DATE]

targets = build_targets()


In [3]:
targets.to_parquet(variables.AGG_TARGET_DATASET_PATH)

In [4]:
%reset -f